### This tutorial is aimed to instill a working understanding of the basics of Reinforcement learning using a popular algorithm (DAPO)

##### High-level process:
1). Use a base LLM (We will use Qwen 2.5 VL) and a dataset containing (x_i, y_i) pairs, where x_i is a prompt and y_i is the corresponding response.
2). for i = 0 to number of iterations:
* (Steps): 
    * Sample a batch of prompts {x_i} (i = 1 to N) from the dataset.
    * For each prompt x_i, generate a response y_i using the base LLM.
    * Compute the reward R_i for each response and normalise it to calculate the advatnage within each group!
    * Create a list of N x G episodes i.e., (x_i, y_i) pairs along with corresponding advantages.
    * Estimate the policy gradient using these episodes.
    * Update the base LLM's parameters using the estimated policy gradient.


#### Environment Setup

We begin by configuring a custom cache directory for Hugging Face models and tokenizers. This ensures that large checkpoints are cached in a consistent location across sessions — particularly useful when working on remote or multi-instance setups.

In [1]:
import os
from pathlib import Path

BASE = Path.home() / "base"
os.environ['HF_HOME'] = str(BASE / "hf_home")

### Import Dependencies

We import core libraries used throughout the training loop:

- `torch`, `transformers`, `datasets`, and `deepspeed` for model loading and RL training.
- `Qwen2.5-VL` is our base vision-language model.
- `vLLM` is used for efficient sampling during rollout generation.
- `wandb` is used for logging and tracking experiment metrics.

> **Note**: Ensure that all packages are installed and GPU is available (`torch.cuda.is_available()`).


In [2]:
#Import dependencies

import torch
import gc
import time
from typing import Optional, Tuple, List, Dict, Union, Any
import deepspeed
import numpy as np
from datasets import load_dataset
from deepspeed import DeepSpeedEngine
from tqdm import trange
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, Qwen2_5_VLForConditionalGeneration
from vllm import LLM, SamplingParams

import wandb

[2025-04-30 15:54:25,696] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/root/miniconda3/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-30 15:54:29,322	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


### Dummy Multi-node Setup for DeepSpeed

DeepSpeed sometimes expects environment variables even in single-GPU setups. To avoid unnecessary errors when using DeepSpeed APIs locally, we manually define the minimal expected distributed environment.

This setup is safe and sufficient for our notebook-based prototype.


In [3]:
#Dummy MultiNode setup - to prevent deepspeed from acting up.
def _find_free_port() -> str:
    import socket, contextlib
    with contextlib.closing(socket.socket(socket.AF_INET, socket.SOCK_STREAM)) as s:
        s.bind(("", 0)); s.listen(1)
        return str(s.getsockname()[1])

os.environ["MASTER_ADDR"]  = "localhost"
os.environ["MASTER_PORT"]  = _find_free_port()
os.environ["RANK"]         = "0"
os.environ["LOCAL_RANK"]   = "0"
os.environ["WORLD_SIZE"]   = "1"


### First lets create some utility functions

In [4]:
#First, lets create some utility functions.
def loadVLLM(model: Union[DeepSpeedEngine, PreTrainedModel], llm: LLM) -> None:
    """
    Loads the model into the VLLM inference engine. It may be either wrapped in DeepSpeed or not.
    Args:
        model (Union[DeepSpeedEngine, PreTrainedModel]): The model to load.
        llm (LLM): The VLLM inference engine.
    Returns:
        None
    """
    StateDict = model.module.state_dict() if isinstance(model, DeepSpeedEngine) else model.state_dict()
    llm.llm_engine.model_executor.driver_worker.model_runner.model.load_weights(StateDict.items())

##### For this tutorial, we will utilise an existing dataset. And inspect a few rows of the dataset. You can present any such dataset, if its in the correct format, to the model.

In [5]:
modelID = 'Qwen/Qwen2.5-VL-3B-Instruct'
datasetName = 'leonardPKU/GEOQA_R1V_Train_8K'
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(
    modelID,
    device_map="cuda",
    torch_dtype=torch.bfloat16
)

dataset = load_dataset(
    datasetName, split="train"
)
#keep the dataset size minimal, as this is merely a tutorial.
dataset = dataset.select(range(0, 500))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
#Show the first three rows.
dataset[:3]

#### Next, we will make the dataset conversational. As Qwen2VL and Qwen2.5 VL are trained on instruct format.

In [6]:
SYSMSG = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)
import PIL
from PIL import Image
from PIL import PngImagePlugin

Template = "{Question}  Output the thinking process in <think> </think> and final answer (number) in <answer> </answer> tags."
def makeConversational(example: Dict[str, List[PngImagePlugin.PngImageFile] | List[str]]) -> Dict[str, List[Dict[str, str | List[Dict[str, str]]]]]:
    return {
            "prompt": [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": Template.format(Question=example["problem"])},
                    ],
                },
            ],
        }

transformedDataset = dataset.map(makeConversational)

In [ ]:
#Examine the transformed dataset - first three rows
transformedDataset[:3]

### 🏆 Reward Functions

In the DAPO framework, reward functions play a pivotal role in guiding the model toward more useful, structured, and correct generations.

We will define **three distinct reward components** based on the original DAPO paper:

1. **Rule-Based Reward (`ruleReward`)**  
   This function returns a reward of:
   - `+1.0` if the generated answer matches the ground truth (via symbolic math parsing or string-based matching).
   - `-1.0` if the answer is incorrect, unparseable, or clearly wrong.
   
2. **Format-Based Reward (`formatReward`)**  
   This evaluates whether the model's output adheres to a specific structure, particularly the presence of `<think> ... </think>` and `<answer> ... </answer>` tags.  
   A well-formatted response gets a reward of `+1.0`, else `0.0`.

3. **Overlength Penalty (`overlongPenalty`)**  
   Responses that exceed a certain length threshold are penalized softly. This discourages unnecessarily verbose or meandering completions.

Finally, the `computeReward` function combines all three components to yield a scalar reward and accompanying diagnostic metrics for each generation.


In [7]:
# Now we can start thinking about Reward functions. The original paper has a rule based reward function, and a format based reward function - which we will implement here.

# The rule based reward function is a simple one - it checks if the answer is correct or not. The format based reward function checks if the answer is in the correct format or not.
from math_verify import verify, parse
def formatReward(completions: str) -> float:
    """
    This function checks if the answer is in the correct format or not
    """
    import re
    pattern = r"<think>.*?</think>\s*<answer>.*?</answer>"
    completionContents = [completion['content'] for completion in completions]
    matches = [re.fullmatch(pattern, content, re.DOTALL) for content in completionContents]
    return [1.0 if match else 0.0 for match in matches]

def ruleReward(completions: str, solution, **kwargs) -> float:
    """
    This function checks if the answer is correct or not
    """
    import re
    import time
    contents = [completion[0]['content'] for completion in completions]
    rewards = []
    currTime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    reward = 0.0
    for content, sol in zip(contents, solution):
        try:
            answer = parse(content) #Symbolic verifier.
            if float(verify(answer, parse(sol))) > 0:
                reward = 1.0 
        except Exception as e:
            pass

        if reward==0.0:
            try:
                solMatch = re.search(r'<answer>(.*?)</answer>', sol)
                groundTruth = solMatch.group(1).strip() if solMatch else sol.strip()

                contentMatch = re.search(r'<answer>(.*?)</answer>', content)
                studentAnswer = contentMatch.group(1).strip() if contentMatch else content.strip()

                if groundTruth == studentAnswer:
                    reward = 1.0
            except Exception:
                reward = -1.0 #Negative reward (as implemented in DAPO, GRPO gives 0 reward for incorrect answers)

        rewards.append(reward)
        if os.getenv("DEBUG_MODE") == 'true':
            logpath = os.getenv("LOG_PATH")
            with open(logpath, 'a') as f:
                f.write(f"------------- {currTime} Accuracy reward: {reward} -------------\n")
                f.write(f"Content: {content}\n")
                f.write(f"Solution: {sol}\n")
    return rewards

#Overlong penalty, as seen in DAPO paper. We punish needlessly long chains which lead to nowhere with a soft-penalty.
def overlongPenalty(
        completionLengths: List[int],
        maxLength: int = 256,
        bufferLength: int = 8,
        penaltyFactor: float = 0.7
) -> List[float]:
    """
    Computes a soft penalty that exceed a certain threshold.
    """
    penalties = []
    thresh = maxLength + bufferLength
    for length in completionLengths:
        excess = length - thresh
        if excess <= 0:
            penalties.append(0.0)
        else:
            penalty = min(penaltyFactor, (excess / maxLength) * penaltyFactor)
            penalties.append(-penalty)
    return penalties

def computeReward(completion: str, 
                  sample: Dict[str, Any], 
                  maxLength: int = 1024, 
                  bufferLength: int = 8, 
                  penaltyFactor: float = 0.7) -> float:
    """
    Computes a reward for a single completion.
    """
    solution = sample['answer']
    formatRewardValue = formatReward(completion)[0]
    ruleRewardValue = ruleReward([completion], solution)[0]

    LenPenalty = overlongPenalty([len(completion)], maxLength, bufferLength, penaltyFactor)[0]
    reward = formatRewardValue + ruleRewardValue + LenPenalty
    metrics = {
        'formatRewardValue': formatRewardValue,
        'ruleRewardValue': ruleRewardValue,
        'LenPenalty': LenPenalty,
    }
    return reward, metrics


###  DAPO Core Components: Advantage Normalization and Token-Level Loss

We now implement the key components that define the **DAPO** (Decoupled Clip and Dynamic Sampling Policy Optimization) algorithm:

1. **Group-normalized advantage computation**  
2. **Token-level clipped policy gradient loss**

---

#### 1. Grouped Advantage Computation

For each prompt $x$, we generate a group of $G$ completions $\{y_i\}_{i=1}^{G}$, each with a scalar reward $R_i$.  
We compute a **z-normalized advantage** for each group:

$$
\hat{A}_i = \frac{R_i - \mu_R}{\sigma_R + \varepsilon}
$$

Where:
- $\mu_R = \frac{1}{G} \sum_{j=1}^{G} R_j$ is the mean reward in the group.
- $\sigma_R$ is the standard deviation of rewards in the group.
- $\varepsilon$ is a small constant to prevent division by zero.

---

#### 2. DAPO Loss Function

DAPO adopts a clipped policy gradient loss inspired by PPO, but applies it **at the token level**.  
This improves gradient signal in long Chain-of-Thought (CoT) completions.

The loss is:

$$
\mathcal{L}_{\text{DAPO}} = \mathbb{E}_{x, y \sim \pi_{\theta_{\text{old}}}} \left[ 
\min\left( 
r_t \hat{A}_t,\;
\text{clip}(r_t,\; 1 - \epsilon_{\text{low}},\; 1 + \epsilon_{\text{high}})\hat{A}_t 
\right)
\right]
$$

Where:
- $r_t = \frac{\pi_\theta(y_t \mid y_{<t}, x)}{\pi_{\theta_{\text{old}}}(y_t \mid y_{<t}, x)}$ is the importance sampling ratio.
- $\hat{A}_t$ is the group-normalized advantage broadcast to each token.

The final loss is averaged over tokens using a mask that excludes padding.

Let's now move on to implementing `dapoAdvantage()` and `DapoLOSS()`.


In [8]:
from trl.core import masked_mean
def dapoAdvantage(rewards: torch.tensor, numGenerations: int, imperceptibleConst:float=1e-4) -> torch.tensor:
    """
    Function which computes the grouped rewards and then returns the advantages in Dapo.
    Arguments:
        rewards: torch.Tensor: Tensor of rewards.
        numGenerations: int: the "G" in GRPO, or number of generations per prompt.
        imperceptibleConst: int: A constant which is added to the denominator to prevent division by zero.
    Returns:
        torch.Tensor: The advantage for DAPO
    """
    rewards = rewards.view(-1, numGenerations)
    meanRewards = torch.mean(rewards, dim=1, keepdim=True)
    stdRewards = torch.std(rewards, dim=1, keepdim=True)
    advantages = (rewards - meanRewards) / (stdRewards + imperceptibleConst)
    return advantages.view(-1)

def DapoLOSS(
        advantages: torch.tensor,
        logProbs: torch.tensor,
        logProbsRef: torch.tensor,
        mask: torch.tensor,
        epsilon_low: float = 0.20,
        epsilon_high: float = 0.28,
        tokenLevel: bool = True,
) -> torch.tensor:
    """
    Objective function for DAPO.
    Arguments:
        advantages: torch.Tensor: The tensor containing advantages.
        logProbs: torch.Tensor: The tensor containing log probabilities (from Policy).
        logProbsRef: torch.Tensor: The tensor containing log probabilities (from Reference).
        mask: torch.Tensor: Completion mask.
        epsilon_low: float: The lower clipping bound
        epsilon_high: float: The upper clipping bound.
        tokenLevel: bool: Whether to compute loss at a token level of a group level. Defaults to True.
    Returns:
        torch.Tensor: The loss for DAPO.
    """
    
    advantages = advantages.unsqueeze(-1).expand_as(logProbs)
    #Compute the ratio
    ratios = torch.exp(logProbs - logProbsRef)

    #PPO-esque clipping objective
    clippedRatios = torch.clamp(ratios, 1 - epsilon_low, 1 + epsilon_high)
    #This helps to preserve gradients
    lossPerToken = torch.exp(logProbs - logProbs.detach()) * advantages
    lossPerToken = -torch.min(advantages * ratios, clippedRatios * advantages)

    #Token level averaging
    if tokenLevel: #true for dapo
        loss = masked_mean(
            lossPerToken,
            mask
        )
    else: #same as grpo aka not token level - but on a group level.
        loss = ((lossPerToken * mask).sum(dim=1) / mask.sum(dim=1)).mean()
    return loss

#### Perhaps a little prescienct, but it would be wise to write the per token log probability function.

In [9]:
def getPerTokenLogPS(
        model: PreTrainedModel, 
        inputIDs: torch.Tensor, 
        attentionMask: torch.Tensor, 
        pixelValues: torch.Tensor,
        imageGridTHW: torch.Tensor
) -> torch.Tensor:
    """
    Computes the Per token log probabilities.
    Arguments:
        model: PreTrainedModel: The model to use.
        inputIDs: torch.Tensor: The input IDs.
        attentionMask: torch.Tensor: The attention mask.
        pixelValues: torch.Tensor: The pixel values.
        imageGridTHW: torch.Tensor: The image grid.
    returns:
        torch.Tensor: The per token log probabilities.
    """
    if imageGridTHW is None:
        logits = model(inputIDs, attention_mask=attentionMask).logits
    else:
        logits = model(inputIDs, attention_mask=attentionMask, pixel_values=pixelValues, image_grid_thw=imageGridTHW).logits
    logits = logits[:, :-1]
    inputIDs = inputIDs[:, 1:]

    #Compute log probability - use a loop to avoid memory issues
    perTokenLogPS = []
    for logitsRow, inputIDsRow in zip(logits, inputIDs):
        logProbs = logitsRow.log_softmax(dim=-1)
        tokenLogProb = torch.gather(logProbs, dim=-1, index=inputIDsRow.unsqueeze(1)).squeeze(1)
        perTokenLogPS.append(tokenLogProb)
    return torch.stack(perTokenLogPS)

### The Final `computeLoss()` Function: Putting It All Together

This function wraps the entire **DAPO training step** — from prompt formatting and generation to computing token-level rewards and policy gradient loss.

---

####  Step-by-Step Breakdown

1. **Preprocessing**:
   - Apply chat templates and tokenize multimodal prompts using the `processor`.
   - Left-pad all sequences to ensure compatibility with Flash Attention.

2. **Prompt Trimming**:
   - If prompt length exceeds `maxLength`, truncate from the left.

3. **Generation**:
   - Generate $G$ completions per prompt using the base model.
   - Identify end-of-sequence tokens and construct attention/completion masks.

4. **Reward Computation**:
   - Wrap completions in a standardized conversational format.
   - Use the `computeReward()` function to assign:
     - **Format reward**
     - **Rule-based reward**
     - **Overlong penalty**
   - Final reward $R_i$ is the sum of the above.

5. **Log-Probability Extraction**:
   - Use `getPerTokenLogPS()` to compute $\log \pi_\theta$ and $\log \pi_{\text{ref}}$ for generated completions.

6. **Advantage Normalization**:
   - Normalize rewards within each group:
   $$
   \hat{A}_i = \frac{R_i - \mu_R}{\sigma_R + \varepsilon}
   $$

7. **DAPO Loss (Token-Level)**:
   - Compute clipped policy gradient loss per token:
   $$
   \mathcal{L}_{\text{DAPO}} = \mathbb{E}\left[
   \min\left(
   r_t \hat{A}_t,\;
   \text{clip}(r_t, 1 - \epsilon_{\text{low}}, 1 + \epsilon_{\text{high}})\hat{A}_t
   \right)
   \right]
   $$
   - Where $r_t$ is the importance sampling ratio:
   $$
   r_t = \frac{\pi_\theta(y_t \mid y_{<t}, x)}{\pi_{\theta_{\text{old}}}(y_t \mid y_{<t}, x)}
   $$

---

This function returns a single scalar loss tensor, ready for `.backward()` in your training loop.


In [10]:
from trl import maybe_apply_chat_template
from transformers import GenerationConfig, PreTrainedModel
import torch

def computeLoss(
        model: PreTrainedModel,
        referenceModel: PreTrainedModel,
        processor: AutoProcessor,
        inputs: list[dict],
        numGenerations: int = 4,
        maxLength: int = 1024,
        epsilonLow: float = 0.2,
        epsilonHigh: float = 0.28,
) -> torch.Tensor:
    device = torch.device("cuda")
    processor.tokenizer.padding_side = "left"

    promptsText = [
        maybe_apply_chat_template(ex, processor)["prompt"]
        for ex in inputs
    ]
    images = [ex['image'] for ex in inputs]
    solutions = [ex['solution'] for ex in inputs]

    #Tokenise and preprocess.
    procInputs = processor(
        text=promptsText,
        images=images,
        return_tensors="pt",
        padding='longest',
        add_special_tokens=False
    )
    for k, v in procInputs.items(): #The prepare inputs in distributed setups.
        if isinstance(v, torch.Tensor):
            procInputs[k] = v.to(device)
    promptIds = procInputs['input_ids']
    promptMask = procInputs['attention_mask']
    pixelValues = procInputs['pixel_values']
    imageGridThw = procInputs['image_grid_thw']

    #Trimming to fit maxlength
    if promptIds.size(1) > maxLength:
        promptIds = promptIds[:, :-maxLength:]
        promptMask = promptMask[:, :-maxLength:]
    
    #Generation config.
    generationConfig = GenerationConfig(
        max_new_tokens=maxLength,
        do_sample=True,
        temperature=1.0,
        top_p=1.0,
        num_return_sequences=numGenerations,
        pad_token_id=processor.tokenizer.pad_token_id,
    )
    genModel = model.module if hasattr(model, 'module') else model
    genKwargs = {
        "input_ids": promptIds,
        "attention_mask": promptMask,
    }
    genKwargs['pixel_values'] = pixelValues
    if imageGridThw is not None:
        genKwargs['image_grid_thw'] = imageGridThw
    
    #Generate and reward
    with torch.no_grad():
        generated = genModel.generate(
            **genKwargs,
            generation_config=generationConfig
        )
        batchSize = promptIds.size(0)
        promptLen = promptIds.size(1)
        totalBatch = batchSize * numGenerations

        completionIds = generated[:, promptLen:]
        promptMaskRepeat = promptMask.repeat_interleave(numGenerations, dim=0)

        isEos = completionIds == processor.tokenizer.eos_token_id
        eosIdx = torch.full(
            (completionIds.size(0),),
            completionIds.size(1),
            dtype=torch.long,
            device=device,
        )
        eosIdx[isEos.any(dim=1)] = (
            isEos.int().argmax(dim=1)[isEos.any(dim=1)]
        )
        seqIdx = torch.arange(
            completionIds.size(1), device=device
        ).unsqueeze(0)
        completionMask = (seqIdx <= eosIdx.unsqueeze(1)).int()

        pixelValuesRepeat = pixelValues.repeat_interleave(numGenerations, dim=0)
        imageGridThwRepeat = (
            imageGridThw.repeat_interleave(numGenerations, dim=0) if imageGridThw is not None else None
        )
        decoded = processor.tokenizer.batch_decode(
            completionIds,
            skip_special_tokens=True,
        )
        completionsWrapped = [[{"role": "assistant", "content": d}] for d in decoded]
        solsRep = [
            sol for sol in solutions for _ in range(numGenerations)
        ]
        rewards = torch.tensor(
            [computeReward(w, {'answer': s})[0] for w, s in zip(completionsWrapped, solsRep)],
            dtype=torch.float32,
            device=device,
        )
    
    #Compute per token logprobs.
    fullInputIds = generated
    fullAttentionMask = torch.cat(
        [promptMaskRepeat, completionMask], dim=1
    )
    logProbs = getPerTokenLogPS(
        model,
        fullInputIds,
        fullAttentionMask,
        pixelValuesRepeat,
        imageGridThwRepeat,
    )[:, promptLen - 1:]
    logProbsReference = getPerTokenLogPS(
        referenceModel,
        fullInputIds,
        fullAttentionMask,
        pixelValuesRepeat,
        imageGridThwRepeat,
    )[:, promptLen - 1:]
    
    #Advantages and Dapo loss.
    advantages = dapoAdvantage(rewards, numGenerations).to(device)
    loss = DapoLOSS(
        logProbs=logProbs,
        logProbsRef=logProbsReference,
        advantages=advantages,
        mask=completionMask,
        epsilon_low=epsilonLow,
        epsilon_high=epsilonHigh,
        tokenLevel=True,
    )
    return loss

##### Login via Wandb to track the experiment

In [11]:
!wandb login

wandb: Currently logged in as: kevindave5735 (kevindave5735-phronetic-ai). Use `wandb login --relogin` to force relogin


### Final Training Loop: DAPO in Action

We now stitch everything together into a complete training loop.

Each iteration performs the following steps:

---

#### 1. Sample Mini-Batch
We randomly sample a batch of prompts and corresponding solutions from the dataset:
- Batch size: $B = 1$
- Generations per prompt: $G = 2$

---

#### 2. Generate Responses & Compute Loss

For each sample in the batch:
- Generate $G$ responses using the current model.
- Use the **reference model** for KL computation.
- Compute the **DAPO loss** using:
  $$
  \mathcal{L}_{\text{DAPO}} = \mathbb{E}\left[
  \min\left(
  r_t \hat{A}_t,\;
  \text{clip}(r_t, 1 - \epsilon_{\text{low}}, 1 + \epsilon_{\text{high}})\hat{A}_t
  \right)
  \right]
  $$
  where $r_t = \frac{\pi_\theta(y_t | x)}{\pi_{\theta_{\text{old}}}(y_t | x)}$ and $\hat{A}_t$ is the normalized advantage.

---

#### 3. Backpropagation & Optimization

We perform:
- `.zero_grad()`
- `.backward()`
- Gradient clipping for stability: $\|\nabla \theta\| \leq 1.0$
- `optimiser.step()` to update model weights.

---

#### 4. Logging & Checkpointing

- Training loss is logged to **Weights & Biases**.
- Every `SAVE_EVERY = 50` iterations, model checkpoints are saved for reproducibility.

---

This is the **end-to-end DAPO training loop**. You can now scale this setup or tweak any components (batch size, prompt sampling logic, reward functions, etc.) to fit your custom dataset and compute.


In [12]:
import gc
import time


def clear_memory():
    # Delete variables if they exist in the current global scope
    if "inputs" in globals():
        del globals()["inputs"]
    if "model" in globals():
        del globals()["model"]
    if "processor" in globals():
        del globals()["processor"]
    if "trainer" in globals():
        del globals()["trainer"]
    if "peft_model" in globals():
        del globals()["peft_model"]
    if "bnb_config" in globals():
        del globals()["bnb_config"]
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


clear_memory()

GPU allocated memory: 0.00 GB
GPU reserved memory: 0.00 GB


In [ ]:
from tqdm import trange
import numpy as np
import deepspeed
import gc
import time
import wandb
import os
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

#You can modify these if you encounter OOM - more specifically, the Generation per sample can be reduced to 2, in case of OOM. You can also try reducing the batch size & max length.
NUM_ITERS = 100
BATCH_SIZE = 1
GENS_PER_SAMPLE = 2
MAX_LEN = 1024
LEARNINGRATE = 1e-5
SAVE_EVERY = 50
#DeepSpeed Config - change the precision to bf16 if you want to use it.
DSCONFIG = {
    "fp16": {"enabled": True},                      
    "zero_optimization": {
        "stage": 2,                                 
        "overlap_comm": False,
    },
    # total episodes per step = BATCH_SIZE * GENS_PER_SAMPLE
    "train_batch_size": BATCH_SIZE * GENS_PER_SAMPLE,
    # micro-batch per GPU = prompts per step
    "train_micro_batch_size_per_gpu": BATCH_SIZE,
    "gradient_accumulation_steps": GENS_PER_SAMPLE,
    "gradient_clipping": 1.0,
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": LEARNINGRATE,
            "betas": (0.9, 0.999),
            "eps": 1e-8,
            "weight_decay": 0.0,
            "torch_adam": True,
        },
    },
}


modelID = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    modelID,
    device_map="cuda",
    torch_dtype=torch.float16,
    attn_implementation='flash_attention_2'
)

referenceModel = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    modelID,
    device_map="cuda",
    torch_dtype=torch.float16,
    attn_implementation='flash_attention_2'
)



#Enable gradient checkpointing for memory optimisation.
model.gradient_checkpointing_enable()
referenceModel.gradient_checkpointing_enable()

optimiser = AdamW(model.parameters(), lr=1e-5)
# scalar = GradScaler()

model = deepspeed.initialize(
    model=model,
    config=DSCONFIG,
    model_parameters=model.parameters(),
    optimizer=optimiser,
)
referenceModel = deepspeed.initialize(
    model=referenceModel,
    config=DSCONFIG,
)



model.config.use_cache = False
referenceModel.config.use_cache = False

processor = AutoProcessor.from_pretrained(modelID)
processor.tokenizer.padding_side = "left"

for iteration in trange(NUM_ITERS):
    print(f"\n===== Iteration {iteration} / {NUM_ITERS} =====")

    #Sample a batch
    indices = np.random.choice(len(transformedDataset), size=BATCH_SIZE, replace=False)
    samples = [transformedDataset[i] for i in indices.tolist()]
    print(samples)

    #Model training step.
    model.train()
    referenceModel.eval()
    if hasattr(referenceModel, "module"): #Check if it is initialised by DeepSpeed.
        referenceModel.module.cpu() #Offload to CPU.
    else:
        referenceModel.cuda()

    #Calls to avoid OOM - if possible.
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    optimiser.zero_grad()
    #Comment the below if you are using deepSpeed.
    # with autocast(device_type="cuda"):
    #     loss = computeLoss(
    #         model=model,
    #         referenceModel=referenceModel,
    #         processor=processor,
    #         inputs=samples,
    #         numGenerations=GENS_PER_SAMPLE,
    #         maxLength=MAX_LEN,
    #         epsilonLow=0.2,
    #         epsilonHigh=0.28,
    #     )
    loss = computeLoss(
        model=model,
        referenceModel=referenceModel,
        processor=processor,
        inputs=samples,
        numGenerations=GENS_PER_SAMPLE,
        maxLength=MAX_LEN,
        epsilonLow=0.2,
        epsilonHigh=0.28,
    )
    loss.backward()
    print(f"DAPO Loss: {loss.item():.4f}")
    model.step()
    

    #Backward prop.
    #optimisations to avoid OOM - comment this out if you are using DeepSpeed as it already does it for you.
    # scalar.scale(loss).backward()
    # scalar.unscale_(optimiser, allowfp16=True)
    # #Gradient clipping to stabilise training
    # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    # scalar.step()
    # scalar.update()

    #Logging.
    wandb.log({
        "iteration": iteration,
        "train/loss": loss.item(),
    })

    print(f"Loss: {loss.item():.4f}")

    #Checkpointing..
    if (iteration + 1) % SAVE_EVERY == 0:
        ckpt_path = f"./checkpoints/dapo_iter_{iteration+1:04d}"
        if hasattr(model, "module"):
            model.module.save_pretrained(f"{ckpt_path}/hf_model")
        model.save_checkpoint(f"{ckpt_path}/deepspeed")
        print(f"Checkpoint saved to {ckpt_path}")

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.42s/it]


[2025-04-30 15:55:37,282] [INFO] [logging.py:128:log_dist] [Rank -1] DeepSpeed info: version=0.15.4, git-hash=unknown, git-branch=unknown
[2025-04-30 15:55:37,282] [INFO] [comm.py:652:init_distributed] cdb=None
[2025-04-30 15:55:37,282] [INFO] [comm.py:683:init_distributed] Initializing TorchBackend in DeepSpeed with backend nccl
[2025-04-30 15:55:37,313] [INFO] [config.py:733:__init__] Config mesh_device None world_size = 1
[2025-04-30 15:55:37,624] [INFO] [logging.py:128:log_dist] [Rank 0] DeepSpeed Flops Profiler Enabled: False
[2025-04-30 15:55:37,627] [INFO] [logging.py:128:log_dist] [Rank 0] Using client Optimizer as basic optimizer
[2025-04-30 15:55:37,627] [INFO] [logging.py:128:log_dist] [Rank 0] Removing param_group that has no 'params' in the basic Optimizer
[2025-04-30 15:55:37,678] [INFO] [logging.py:128:log_dist] [Rank 0] DeepSpeed Basic Optimizer = AdamW
[2025-04-30 15:55:37,678] [INFO] [utils.py:59:is_zero_supported_optimizer] Checking ZeRO support for optimizer=AdamW t

OutOfMemoryError: CUDA out of memory. Tried to allocate 13.99 GiB. GPU 0 has a total capacity of 47.33 GiB of which 10.50 GiB is free. Including non-PyTorch memory, this process has 36.82 GiB memory in use. Of the allocated memory 27.97 GiB is allocated by PyTorch, and 8.23 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)